In [3]:
#Importing the needed Libaries
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier

In [4]:
#Loading the Training dataset
train_path = r"F:\3000 stuff\UNSW_NB15_training-set.csv"

df_train = pd.read_csv(train_path).drop_duplicates().reset_index(drop=True)

df_train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [5]:
#Remove non feature colums and reduce the risk of leaks
drop_cols = ["srcip", "dstip", "attack_cat", "id"]
df_train = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns], errors="ignore")

y_train = df_train["label"].astype(int)
X_train = df_train.drop(columns=["label"])

In [6]:
#Data Preprocessing
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med).replace([np.inf, -np.inf], med)

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)

In [7]:
#Ensuring the data remains numeric
X_train_enc = X_train_enc.astype('float64')

In [8]:
#Save the feature columns to a list
feature_cols = X_train_enc.columns.tolist()

In [9]:
#Training Model
rf_model = RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        class_weight='balanced', n_jobs=-1, random_state=42
)

rf_model.fit(X_train_enc, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_split=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [10]:
#Saving the model and the feature schema
joblib.dump(rf_model, r"F:\3000 stuff\rf_unsw_nb15.pkl")
joblib.dump(X_train_enc.columns.tolist(), r"F:\3000 stuff\rf_feature_columnsv2.pkl")

['F:\\3000 stuff\\rf_feature_columnsv2.pkl']